# 工具评测

由多个 agent 各自独立运行评测文件中的单个评测任务。

In [20]:
import json
import re
import time
import traceback
import xml.etree.ElementTree as ET  # noqa: S314
from pathlib import Path
from typing import Any

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

## 提示词

In [21]:
# 内嵌的评测提示词
EVALUATION_PROMPT = """你是一个可以使用工具的 AI 助手。

收到任务后，你必须：
1. 使用可用的工具完成任务
2. 总结你做法中的每一步，用 <summary> 标签包住
3. 对所提供的工具给出反馈，用 <feedback> 标签包住
4. 给出最终回答，用 <response> 标签包住

总结要求：
- 在 <summary> 标签里必须说明：
  - 完成任务所经过的步骤
  - 用了哪些工具、按什么顺序、为什么
  - 给每个工具传了什么输入
  - 每个工具返回了什么输出
  - 你是怎样得出最终回答的

反馈要求：
- 在 <feedback> 标签里对工具提出有建设性的反馈：
  - 评价工具名：是否清楚、是否能说明用途？
  - 评价输入参数：文档是否充分？必填和可选参数是否分得清？
  - 评价描述：是否准确说明了工具的功能？
  - 评价使用工具时遇到的错误：工具是否执行失败？返回的 token 是否太多？
  - 指出具体的改进点，并解释为什么这样改有帮助
  - 建议要具体、可执行

回答要求：
- 回答要简洁，直接针对所问的内容
- 最终回答一律放在 <response> 标签里
- 如果无法完成任务，返回 <response>NOT_FOUND</response>
- 数值型答案只给数字
- ID 只给 ID
- 名称或文本给出所要求的原文
- 最终回答放在最后"""

## Agent 循环

In [ ]:
client = Anthropic(base_url="https://api.deepseek.com/anthropic")
model = "deepseek-v4-flash"


def agent_loop(prompt: str, tools: list[dict[str, Any]] = None) -> tuple[str, dict[str, Any]]:
    """用于工具评测的简化版 agent"""
    # 循环每轮只处理一个 tool_use，所以关掉并行调用，让模型每轮最多发一个
    messages = [{"role": "user", "content": prompt}]

    response = client.messages.create(
        model=model,
        max_tokens=4096,
        system=EVALUATION_PROMPT,
        messages=messages,
        tools=tools,
        tool_choice={"type": "auto", "disable_parallel_tool_use": True},
    )

    messages.append({"role": "assistant", "content": response.content})

    # 记录工具调用次数和耗时
    tool_metrics = {}  # {tool_name: {"count": N, "durations": [X1, X2, ...]}}

    def _prepare_tool_result(tool_use_id, tool_result):
        return {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": tool_result,
                }
            ],
        }

    while response.stop_reason == "tool_use":
        tool_use = next(block for block in response.content if block.type == "tool_use")
        tool_name = tool_use.name

        tool_start_ts = time.time()
        try:
            # 注意：这里用 eval 动态调用工具函数只是为了演示。
            # 生产环境请用更安全的分发方式，比如函数字典。
            tool_response = eval(  # noqa: S307
                f"{tool_name}(**tool_use.input)"
            )  # 用模型给的输入调用工具函数
        except Exception as e:
            tool_response = f"执行工具 {tool_name} 出错：{str(e)}\n"
            tool_response += traceback.format_exc()
        tool_duration = time.time() - tool_start_ts

        # 更新工具指标
        if tool_name not in tool_metrics:
            tool_metrics[tool_name] = {"count": 0, "durations": []}
        tool_metrics[tool_name]["count"] += 1
        tool_metrics[tool_name]["durations"].append(tool_duration)

        # 组装工具结果并追加到 messages
        messages.append(_prepare_tool_result(tool_use.id, tool_response))
        response = client.messages.create(
            model=model,
            max_tokens=4096,
            system=EVALUATION_PROMPT,
            messages=messages,
            tools=tools,
            tool_choice={"type": "auto", "disable_parallel_tool_use": True},
        )
        messages.append({"role": "assistant", "content": response.content})

    response = next(
        (block.text for block in response.content if hasattr(block, "text")),
        None,
    )
    return response, tool_metrics

## 辅助函数

In [15]:
def parse_evaluation_file(file_path: Path) -> list[dict[str, Any]]:
    """解析 XML 评测文件，返回评测任务列表。"""
    try:
        # 解析可信的本地 XML 评测文件
        tree = ET.parse(file_path)  # noqa: S314
        root = tree.getroot()
        evaluations = []

        # 查找 task 元素
        tasks = root.findall(".//task")
        for task in tasks:
            prompt_elem = task.find("prompt")
            response_elem = task.find("response")

            if prompt_elem is not None and response_elem is not None:
                eval_dict = {
                    "prompt": (prompt_elem.text or "").strip(),
                    "response": (response_elem.text or "").strip(),
                }
                evaluations.append(eval_dict)

        return evaluations
    except Exception as e:
        print(f"解析评测文件 {file_path} 出错：{e}")
        return []

In [16]:
def evaluate_single_task(
    task: dict[str, Any], tools: list[dict[str, Any]], task_index: int
) -> dict[str, Any]:
    """用给定的工具评测单个任务。"""
    start_time = time.time()

    # 运行任务
    print(f"任务 {task_index + 1}：运行任务，提示词：{task['prompt']}")
    response, tool_metrics = agent_loop(task["prompt"], tools)

    # 提取所有带标签的内容
    def _extract_xml_content(text, tag):
        pattern = rf"<{tag}>(.*?)</{tag}>"
        matches = re.findall(pattern, text, re.DOTALL)
        return matches[-1].strip() if matches else None

    response, summary, feedback = (
        _extract_xml_content(response, tag) for tag in ["response", "summary", "feedback"]
    )
    duration_seconds = time.time() - start_time

    return {
        "prompt": task["prompt"],
        "expected": task["response"],
        "actual": response,
        "score": int(response == task["response"]),
        "total_duration": duration_seconds,
        "tool_calls": tool_metrics,
        "num_tool_calls": sum(len(metrics["durations"]) for metrics in tool_metrics.values()),
        "summary": summary,
        "feedback": feedback,
    }

## 主评测函数

In [17]:
# 报告模板
REPORT_HEADER = """
# 评测报告

## 概览

- **准确率**： {correct}/{total} ({accuracy:.1f}%)
- **平均任务耗时**： {average_duration_s:.2f}s
- **平均每个任务的工具调用次数**： {average_tool_calls:.2f}
- **工具调用总次数**： {total_tool_calls}

---
"""

TASK_TEMPLATE = """
### 任务

**提示词**： {prompt}
**标准答案**： `{expected_response}`
**实际回答**： `{actual_response}`
**是否正确**： {correct_indicator}
**耗时**： {total_duration:.2f}s
**工具调用**： {tool_calls}

**总结**
{summary}

**反馈**
{feedback}

---
"""


def run_evaluation(eval_path: str, tools: list[dict[str, Any]]) -> str:
    """
    用提供的工具跑一遍评测，就是一个简单的循环。

    Args:
        eval_path: XML 评测文件路径
        tools: 评测时使用的工具定义列表

    """
    print("🚀 开始评测")

    eval_file = Path(eval_path)

    # 解析评测任务
    tasks = parse_evaluation_file(eval_file)

    print(f"📋 已加载 {len(tasks)} 个评测任务")

    # 简单循环跑完所有任务
    results = []
    for i, task in enumerate(tasks):
        print(f"处理任务 {i + 1}/{len(tasks)}")
        results.append(evaluate_single_task(task, tools, i))

    # 计算汇总统计
    correct = sum(r["score"] for r in results)
    accuracy = (correct / len(results)) * 100
    average_duration_s = sum(r["total_duration"] for r in results) / len(results)
    average_tool_calls = sum(r["num_tool_calls"] for r in results) / len(results)
    total_tool_calls = sum(r["num_tool_calls"] for r in results)

    report = REPORT_HEADER.format(
        correct=correct,
        total=len(results),
        accuracy=accuracy,
        average_duration_s=average_duration_s,
        average_tool_calls=average_tool_calls,
        total_tool_calls=total_tool_calls,
    )

    report += "".join(
        [
            TASK_TEMPLATE.format(
                prompt=task["prompt"],
                expected_response=task["response"],
                actual_response=result["actual"],
                correct_indicator="✅" if result["score"] else "❌",
                total_duration=result["total_duration"],
                tool_calls=json.dumps(result["tool_calls"], indent=2),
                summary=result["summary"] or "N/A",
                feedback=result["feedback"] or "N/A",
            )
            for task, result in zip(tasks, results, strict=False)
        ]
    )
    # 拼接所有部分得到最终报告
    return report

## 计算器工具

In [18]:
def calculator(expression: str) -> str:
    """执行算术运算的基础计算器。"""
    try:
        # 注意：这里用限制了 builtins 的 eval 只是为了演示。
        # 生产环境请用更安全的替代方案，比如数学表达式解析器。
        result = eval(expression, {"__builtins__": {}}, {})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"错误：{str(e)}"


# 定义计算器的工具 schema
calculator_tool = {
    "name": "calculator",
    "description": "",  # 故意写得没用的工具描述。
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "",  # 故意写得没用的 schema 描述。
            }
        },
        "required": ["expression"],
    },
}

# 设置工具列表
tools = [calculator_tool]

## 运行评测

In [19]:
# 运行评测
print("✅ 使用计算器工具")

report = run_evaluation(eval_path="evaluation.xml", tools=tools)

print(report)

✅ 使用计算器工具
🚀 开始评测
📋 已加载 8 个评测任务
处理任务 1/8
任务 1：运行任务，提示词：10,000 美元按 5% 年利率、每月复利投资 3 年，计算复利。最终金额是多少美元（保留 2 位小数）？


TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"